Rag Pipeline:Data ingestion to vector db pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from pathlib import Path

/tmp/ipykernel_10065/2775815901.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader


Read all the pdfs inside the directory :

In [2]:
def process_all_pdfs(pdf_directory):
    """Process all pdf files in a directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"found{len(pdf_files)}PDF files to process")
    for pdf_file in pdf_files:
        print(f"\nProcessing:{pdf_file.name}")
        try :
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
                all_documents.extend(documents)
                print(f"Loaded {len(documents)} pages")
        except Exception  as e:
            print(f"Error{e}")
        print(f"\nTotal documents loaded:{len(all_documents)}")
        return all_documents
all_pdf_documents=process_all_pdfs("../data")

found1PDF files to process

Processing:Tableau_Derivees.pdf
Loaded 4 pages
Loaded 4 pages
Loaded 4 pages
Loaded 4 pages

Total documents loaded:16


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2015-07-25T11:25:12+02:00', 'author': 'Clara Parfenoff', 'moddate': '2015-08-02T07:36:52+02:00', 'subject': 'Tableau de dérivées', 'title': 'Tableau de dérivées', 'source': '../data/pdf/Tableau_Derivees.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Tableau_Derivees.pdf', 'file_type': 'pdf'}, page_content='Tableau de dérivées \nI) Dérivées des fonctions usuelles.  \n𝑭𝒐𝒏𝒄𝒕𝒊𝒐𝒏 𝒇 ∶ 𝑫é𝒓𝒊𝒗𝒂𝒃𝒍𝒆 𝒔𝒖𝒓: 𝑭𝒐𝒏𝒄𝒕𝒊𝒐𝒏 𝒅é𝒓𝒊𝒗é𝒆 𝒇’ ∶ \n𝒇(𝒙)  =  𝒌 ( 𝒌 \uf0ce  ℝ  ) ℝ 𝒇’(𝒙)  =  𝟎 \n𝒇(𝒙)  =  𝒙 ℝ 𝒇’(𝒙)  =  𝟏 \n𝒇(𝒙)  =  𝒙𝟐 ℝ 𝒇’(𝒙)  =  𝟐𝒙 \n𝒇(𝒙)  =  𝒙 𝒏( 𝒏 \uf0ce \uf04e ) ℝ 𝒇’(𝒙)  =  𝒏𝒙 𝒏−𝟏 \n𝒇(𝒙)  =  \n𝟏\n𝒙 ]- ∞ ; 0[∪ ]0 ; + ∞[ 𝒇’(𝒙) = −\n𝟏\n𝒙𝟐 \n𝒇(𝒙)  = √𝒙 ]0 ; + ∞[ 𝒇’(𝒙)  =  𝟏\n𝟐√𝒙 \n𝒇(𝒙) = 𝟏\n𝒙𝒏 ℝ ∖ {0} 𝒇’(𝒙) = −\n𝒏\n𝒙𝒏+𝟏 \n𝒇(𝒙)  =  𝒔𝒊𝒏(𝒙) ℝ 𝒇’(𝒙)  =  𝒄𝒐𝒔(𝒙) \n𝒇(𝒙)  =  𝒄𝒐𝒔(𝒙) ℝ 𝒇’(𝒙) =  −𝒔𝒊𝒏(𝒙) \n𝒇(𝒙)  =  𝒍𝒏(𝒙) ]0 ;+∞[ 𝒇’(𝒙) = \n𝟏\n𝒙  \n𝒇(𝒙) = 𝒆𝒙 ℝ 𝒇’(𝒙) = 𝒆𝒙'),
 Document(metada

Text splitting get into chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                                chunk_overlap=chunk_overlap,
                                                length_function=len,
                                                separators=["\n\n","\n","",""])
    split_docs=text_splitter.split_documents(documents)
    print(f"Split{len(documents)} documents into {len(split_docs)} chunks")
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content {split_docs[0].page_content[:200]}...")
        print(f"Metadata:{split_docs[0].metadata}")
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split16 documents into 20 chunks

Example chunk:
Content Tableau de dérivées 
I) Dérivées des fonctions usuelles.  
𝑭𝒐𝒏𝒄𝒕𝒊𝒐𝒏 𝒇 ∶ 𝑫é𝒓𝒊𝒗𝒂𝒃𝒍𝒆 𝒔𝒖𝒓: 𝑭𝒐𝒏𝒄𝒕𝒊𝒐𝒏 𝒅é𝒓𝒊𝒗é𝒆 𝒇’ ∶ 
𝒇(𝒙)  =  𝒌 ( 𝒌   ℝ  ) ℝ 𝒇’(𝒙)  =  𝟎 
𝒇(𝒙)  =  𝒙 ℝ 𝒇’(𝒙)  =  𝟏 
𝒇(𝒙)  =  𝒙𝟐 ℝ 𝒇’(𝒙)  =  𝟐𝒙...
Metadata:{'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2015-07-25T11:25:12+02:00', 'author': 'Clara Parfenoff', 'moddate': '2015-08-02T07:36:52+02:00', 'subject': 'Tableau de dérivées', 'title': 'Tableau de dérivées', 'source': '../data/pdf/Tableau_Derivees.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Tableau_Derivees.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2015-07-25T11:25:12+02:00', 'author': 'Clara Parfenoff', 'moddate': '2015-08-02T07:36:52+02:00', 'subject': 'Tableau de dérivées', 'title': 'Tableau de dérivées', 'source': '../data/pdf/Tableau_Derivees.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Tableau_Derivees.pdf', 'file_type': 'pdf'}, page_content='Tableau de dérivées \nI) Dérivées des fonctions usuelles.  \n𝑭𝒐𝒏𝒄𝒕𝒊𝒐𝒏 𝒇 ∶ 𝑫é𝒓𝒊𝒗𝒂𝒃𝒍𝒆 𝒔𝒖𝒓: 𝑭𝒐𝒏𝒄𝒕𝒊𝒐𝒏 𝒅é𝒓𝒊𝒗é𝒆 𝒇’ ∶ \n𝒇(𝒙)  =  𝒌 ( 𝒌 \uf0ce  ℝ  ) ℝ 𝒇’(𝒙)  =  𝟎 \n𝒇(𝒙)  =  𝒙 ℝ 𝒇’(𝒙)  =  𝟏 \n𝒇(𝒙)  =  𝒙𝟐 ℝ 𝒇’(𝒙)  =  𝟐𝒙 \n𝒇(𝒙)  =  𝒙 𝒏( 𝒏 \uf0ce \uf04e ) ℝ 𝒇’(𝒙)  =  𝒏𝒙 𝒏−𝟏 \n𝒇(𝒙)  =  \n𝟏\n𝒙 ]- ∞ ; 0[∪ ]0 ; + ∞[ 𝒇’(𝒙) = −\n𝟏\n𝒙𝟐 \n𝒇(𝒙)  = √𝒙 ]0 ; + ∞[ 𝒇’(𝒙)  =  𝟏\n𝟐√𝒙 \n𝒇(𝒙) = 𝟏\n𝒙𝒏 ℝ ∖ {0} 𝒇’(𝒙) = −\n𝒏\n𝒙𝒏+𝟏 \n𝒇(𝒙)  =  𝒔𝒊𝒏(𝒙) ℝ 𝒇’(𝒙)  =  𝒄𝒐𝒔(𝒙) \n𝒇(𝒙)  =  𝒄𝒐𝒔(𝒙) ℝ 𝒇’(𝒙) =  −𝒔𝒊𝒏(𝒙) \n𝒇(𝒙)  =  𝒍𝒏(𝒙) ]0 ;+∞[ 𝒇’(𝒙) = \n𝟏\n𝒙  \n𝒇(𝒙) = 𝒆𝒙 ℝ 𝒇’(𝒙) = 𝒆𝒙'),
 Document(metada

Embedding and vectorStoredb

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager:
        Args:
        model_name:HuggingFace model for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully.Embedding dimension:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading the model {self.model_name}:e")
            raise
    def generate_embeddings(self,texts:List[str])->np.ndarray:
        """Generate embeddings for a list of texts
        Args:
        texts:List of text strings to embed
        Returns:
        numpy array of embedding with shape (len(texts),embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generate embeddings with shape:{embeddings.shape}")
        return embeddings
####Initialize the embedding manager:
embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model:all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully.Embedding dimension:384


/tmp/ipykernel_10065/2425892475.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully.Embedding dimension:{self.model.get_sentence_embedding_dimension()}")


Vectorstore:

In [11]:
class VectorStore:
    """Manages document embeddings in a chromadb vector sotre"""
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="../data/vectore_store"):
        """Initialize the vector store:
        Args:
        collection_name:Name of the chromadb collection
        persist_directory:Directory to persist the vectror store:
        """
        self.collection_name=collection_name
        self.persist_directory=persist_direcotry
        self.client=None,
        self.collection=None
        self._initialize_store()
    def _initialize_store(self):
        """Initialize Chromadb client and collection"""
        try:
            #Create persistent chromadb client:
            os.makedir(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            #Get or create collection:
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embedding for Rag"})
            print(f"Vector store initialized .Collection:{self.collection_name}")
            print(f"Existing documents in collection:{self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store:{e}")
            raise
            
     